# Simulating Circuit

This page runs circuits on the local simulator and reads back what they produced, including the quantum
state each shot left behind and its Wigner function.

The simulator needs no access to the machine, and it can hold states the machine cannot prepare, so it
is where a circuit is usually tried first. The [Quickstart](quickstart.ipynb) is the shortest version of
this page; here we take a longer circuit and go through the client's options.

(sec:simulation-configuring-circuit-representation)=

## Configuring circuit representation

First, we create the teleportation circuit below.

```{image} _images/circuit_repr_teleportation_light.svg
:alt: teleportation
:class: only-light
```
```{image} _images/circuit_repr_teleportation_dark.svg
:alt: teleportation
:class: only-dark
```

In [ ]:
# Draw the figures on this page as SVG so that they stay sharp at any zoom level.
# The cell itself is hidden from the built page by the `remove-cell` tag.
%config InlineBackend.figure_formats = ["svg"]

In [ ]:
from math import pi

import numpy as np

from claveles.circuit import CircuitRepr
from claveles.circuit.ops.std import BeamSplitter, Displacement, MeasureHomodyne
from claveles.circuit.state import QuantumState
from claveles.circuit.visualize import make_figure

mode0_x = 3.0
mode0_p = -1.5

# The states are named here rather than only placed, so that the next section can draw them without
# reading them back off the circuit.
# The state to teleport: the vacuum, displaced to (x, p).
input_state = QuantumState.gaussian(
    np.array([mode0_x + 0.0j, mode0_p + 0.0j]),
    QuantumState.vacuum().cov,
)
# The pair that carries it across.
p_squeezed = QuantumState.squeezed(r=1.5, phi=pi / 2)
x_squeezed = QuantumState.squeezed(r=1.5, phi=0)

# Construct the teleportation circuit. A state is placed on a mode by the first operation that
# touches it, so the placements come before the gates.
circuit = CircuitRepr("teleportation")
circuit.Q(0) | input_state
circuit.Q(1) | p_squeezed
circuit.Q(2) | x_squeezed

circuit.Q(1, 2) | BeamSplitter(theta=pi / 4, phi=pi)
circuit.Q(0, 1) | BeamSplitter(theta=pi / 4, phi=pi)
# Measure modes 0 and 1. Measuring a mode consumes it, so only mode 2 is left afterwards.
m0 = circuit.Q(0) | MeasureHomodyne(phi=0)
m1 = circuit.Q(1) | MeasureHomodyne(phi=pi / 2)
# Apply displacement with feedforward.
circuit.Q(2) | Displacement(m0 - 1j * m1)

make_figure(circuit);

### Visualizing the initial state modes

A {py:class}`~claveles.circuit.state.QuantumState` can be visualized as a Wigner function. Two methods on it are the way in:

* {py:meth}`~claveles.circuit.state.QuantumState.draw_wigner`: Plot the Wigner function of a state holding one mode, over a mesh of the quadrature variables x and p. {py:meth}`~claveles.circuit.state.QuantumState.extract_mode` is how a mode of a wider state is taken out first, and the figure is returned so that it can be adjusted or saved.
* {py:meth}`~claveles.circuit.state.QuantumState.wigner_func`: Return the Wigner function itself, of the $2n$ quadratures of an $n$-mode state, which is what to reach for when a few values are wanted instead of a plot.

```{seealso}
The {py:mod}`~claveles.circuit.visualize_state` module holds the functions underneath these, which take one basic state rather than a container: {py:func}`~claveles.circuit.visualize_state.meshed_wigner`, {py:func}`~claveles.circuit.visualize_state.make_wigner_figure`, {py:func}`~claveles.circuit.visualize_state.wigner_values` and {py:func}`~claveles.circuit.visualize_state.save_wigner_plot`.
```

Now let's visualize the Wigner function of each mode in the circuit.

In [ ]:
from numpy import linspace

input_state.draw_wigner(xvec=linspace(-10, 10, 401), pvec=linspace(-10, 10, 401));

In [ ]:
p_squeezed.draw_wigner(xvec=linspace(-10, 10, 401), pvec=linspace(-10, 10, 401));

In [ ]:
x_squeezed.draw_wigner(xvec=linspace(-10, 10, 401), pvec=linspace(-10, 10, 401));

## Configuring the client

A client says where a program runs; {py:class}`~claveles.client.SimulatorClient` is the one that simulates
it. See {ref}`Clients <sec:clients>` for how it compares with the machine's client and for the errors
both of them raise.

````{note}
{py:class}`~claveles.client.SimulatorClient` requires the `claveles_simulator` package, which the
`claveles` meta-package installs by default — see [Installation](installation.md). It can simulate all three
representations.
````

Let's create a {py:class}`~claveles.client.SimulatorClient` instance as shown below.

In [ ]:
from claveles.client import SimulatorClient, SimulatorClientResult

client = SimulatorClient()

(sec:simulation-client-options)=

### The client's options

Every option can be given to the constructor or set on the client afterwards. Most of them are read by
only part of the simulator, and one given where it is not read reports an
{py:class}`~claveles.warnings.IgnoredOptionWarning` rather than being dropped in silence.

| Option | Default | Read by | Description |
| --- | --- | --- | --- |
| {py:attr}`~claveles.client.abstract.AbstractClient.n_shots` | `1024` | every run | How many times the program is executed. |
| {py:attr}`~claveles.client.simulator_client.SimulatorClient.backend` | `None` | every run | Which backend to run on. `None` behaves like `"cpu"` locally, and lets the server choose for `mode="remote"` and `mode="simulator-server"`. The {ref}`Support Matrix <sec:support-backends>` lists them. |
| {py:attr}`~claveles.client.simulator_client.SimulatorClient.mode` | `"local"` | every run | Whether the simulation runs in this process or on a Cloud Simulator Server; see {ref}`sec:simulation-where-it-runs`. |
| `random_seed` | `None` | every run | Random seed. `None` draws a fresh random seed for each run, so two seedless runs sample independently; pass an explicit seed to reproduce a run. For `mode="local"` the chosen seed is not returned; for `mode="remote"` (the production path through the Cloud User API) the seed is randomized on the server but is not relayed back to the client either, so pass an explicit seed to reproduce such a run. Within a run, each shot draws from its own stream derived from the seed and the shot index. |
| {py:attr}`~claveles.client.simulator_client.SimulatorClient.state_save_policy` | `None` | circuits | Which shots' states to keep: `"none"`, `"first_only"` or `"all"`. `None` resolves to `"none"` locally and `"first_only"` on the server. |
| {py:attr}`~claveles.client.simulator_client.SimulatorClient.state_representation` | `"auto"` | circuits | How the state is held — `"auto"`, `"bosonic"` or `"fock"`; see below. |
| `decompose_std` | `False` | circuits | When `True`, {py:mod}`~claveles.circuit.ops.std` operations are lowered to their intrinsic decomposition before the circuit is simulated, instead of being simulated natively where a dedicated implementation exists. |
| {py:attr}`~claveles.client.simulator_client.SimulatorClient.resource_squeezing_level` | `None` | graph, machinery | Squeezing level of the resource states, in dB. `None` resolves to {py:data}`~claveles.client.simulator_client.DEFAULT_RESOURCE_SQUEEZING_LEVEL` (10.0). A higher level means less noise from the imperfection of the EPR state; see {ref}`sec:states-epr`. |
| {py:attr}`~claveles.client.simulator_client.SimulatorClient.max_bond_dim`, {py:attr}`~claveles.client.simulator_client.SimulatorClient.svd_cutoff`, {py:attr}`~claveles.client.simulator_client.SimulatorClient.track_truncation_error`, {py:attr}`~claveles.client.simulator_client.SimulatorClient.max_saved_state_elements` | — | Fock circuits | The accuracy and memory bounds of the Fock backend; see {ref}`sec:simulation-fock-options`. |
| {py:attr}`~claveles.client.simulator_client.SimulatorClient.url`, {py:attr}`~claveles.client.simulator_client.SimulatorClient.api_key`, {py:attr}`~claveles.client.simulator_client.SimulatorClient.timeout_msec`, {py:attr}`~claveles.client.simulator_client.SimulatorClient.polling_interval` | — | `mode="remote"`, `mode="simulator-server"` | Where that server is and how it is polled; see {ref}`sec:simulation-where-it-runs`. |
| {py:attr}`~claveles.client.simulator_client.SimulatorClient.timezone` | UTC | every run | Timezone of the timestamps in {py:attr}`~claveles.client.SimulatorClientResult.execution_details`. |

The options that shape a run the most are covered in the sections below. The rest of this page uses
{py:attr}`~claveles.client.abstract.AbstractClient.n_shots` and
{py:attr}`~claveles.client.simulator_client.SimulatorClient.state_save_policy`.

You can specify how many times the circuit will be executed by setting the
{py:attr}`~claveles.client.abstract.AbstractClient.n_shots` attribute.

In [ ]:
# Run the circuit for hundred times
client.n_shots = 100

{py:attr}`~claveles.client.simulator_client.SimulatorClient.state_save_policy` controls whether the quantum
state a shot leaves behind is saved:

- `"none"`: do not save any states. This is what `None` resolves to for `mode="local"`.
- `"first_only"`: save the state of the first shot only.
- `"all"`: save the states of all shots.

A saved state is read off the shot, as {ref}`sec:simulation-reading-results` describes.

````{note}
A circuit that contains no measurement is **deterministic**, so every shot leaves the same state behind.
For such a circuit `n_shots=1` with `state_save_policy="all"` is all that is needed.
````

In [ ]:
client.state_save_policy = "all"

(sec:simulation-where-it-runs)=

### Where the simulation runs

{py:attr}`~claveles.client.simulator_client.SimulatorClient.mode` chooses that, and it is what the
`mode="local"` in the option tables on this page refers to.

| Value | Behaviour |
| --- | --- |
| `"local"` (default) | Run in this process, with the `claveles_simulator` wheel. Nothing leaves the machine. |
| `"remote"` | Submit through the Cloud User API — the same endpoint and API key {py:class}`~claveles.client.OptQCClient` uses with `target="cloud"` — which relays the job to a Cloud Simulator Server. This is the production remote path, and it needs {py:attr}`~claveles.client.simulator_client.SimulatorClient.url` and {py:attr}`~claveles.client.simulator_client.SimulatorClient.api_key`. |
| `"simulator-server"` | Submit directly to a running Cloud Simulator Server over HTTP, which needs {py:attr}`~claveles.client.simulator_client.SimulatorClient.url` and {py:attr}`~claveles.client.simulator_client.SimulatorClient.api_key`. This is an internal verification and development path that bypasses the Cloud User API. It is not available to general users, who reach the server through `"remote"` instead. |

`"remote"` and `"simulator-server"` both run on a Cloud Simulator Server, so two things differ from
`"local"` for either: the server applies its own
{py:attr}`~claveles.client.simulator_client.SimulatorClient.max_saved_state_elements`, and it fixes
{py:attr}`~claveles.client.simulator_client.SimulatorClient.state_save_policy` at
{py:data}`~claveles.client.simulator_client.SIMULATOR_SERVER_STATE_SAVE_POLICY`, so setting either one
is refused rather than ignored.

Those routes also have three errors of their own, alongside the hierarchy every client shares; all of
them are described in {ref}`sec:clients-errors`.

## Choosing the state representation

The simulator holds a circuit's state in one of two ways, and which one it uses decides what the circuit can do.

- The **bosonic** representation holds the state as a weighted sum of Gaussians in phase space. Gaussian operations are exact, and every backend can run it.
- The **Fock** representation holds it in the photon-number basis, truncated at a finite dimension per mode. It is the only one that can apply a non-Gaussian gate such as {py:class}`std.Kerr <claveles.circuit.ops.std.Kerr>` or answer a photon-number measurement ({py:class}`std.MeasureFock <claveles.circuit.ops.std.MeasureFock>`).

{py:attr}`~claveles.client.simulator_client.SimulatorClient.state_representation` chooses between them.

| Value | Behaviour |
| --- | --- |
| `"auto"` (default) | Run in whichever representation the circuit needs. A circuit that requires the Fock representation anywhere is run in it. |
| `"bosonic"` | Refuse a circuit that would need the Fock representation, rather than converting its states into it. Use this to be sure a run stays Gaussian. |
| `"fock"` | Convert every bosonic state into the Fock basis and run the whole circuit there, even where `"auto"` would have stayed bosonic. Use this to compare the same circuit against its exact bosonic result. |

A circuit that needs neither is unaffected by this setting: nothing is converted, and the result and the speed are the same as before.

```{note}
**`"auto"` converts the bosonic states before the first operation, not at the operation that needs it.**

A circuit that starts from bosonic states and reaches a non-Gaussian gate partway through is converted up front, so the operations *before* that gate are applied in the truncated Fock basis rather than exactly. That costs accuracy but runs the circuit rather than refusing it.

Converting at the operation that needs it is not implemented yet. When it is, results for such circuits will shift slightly, because the operations before the gate will then be applied exactly. Circuits that need no conversion at all are not affected.
```

### The dimension the conversion uses

Converting a bosonic state into the Fock basis needs a dimension for each mode. Give it when placing the state:

```python
circuit.Q(0) | QuantumState.cat(alpha=1.0, cutoff_dim=80)
```

Leave it out and the simulator chooses one, reporting that it did through {py:class}`~claveles.warnings.StateConversionWarning`. No single dimension suits every state — a cat state is exact by 8 while a GKP state needs at least 20 — so the choice is made against the state at hand rather than from a fixed default.

A chosen dimension is a guess about your state, so it may be one you would rather not have made silently. Promoting that warning to an error refuses the run instead, and the message names the modes it was about to convert:

In [ ]:
import warnings

from claveles.execute import execute
from claveles.warnings import StateConversionWarning

no_cutoff = CircuitRepr("no_cutoff")
no_cutoff.Q(0) | QuantumState.cat(alpha=1.0)  # no cutoff_dim given
no_cutoff.Q(0) | MeasureHomodyne(phi=0.0)

with warnings.catch_warnings():
    warnings.simplefilter("error", StateConversionWarning)
    try:
        execute(no_cutoff, SimulatorClient(n_shots=1, state_representation="fock"))
    except StateConversionWarning as warning:
        print("StateConversionWarning:", warning)

Whatever the dimension, the conversion truncates each state's infinite Fock tail. The weight lost that way is left in the state rather than normalized away, the same convention the rest of the Fock representation follows, so a converted state can arrive with a norm below 1.

### What a conversion cannot do

- **A GPU backend cannot run a circuit that needs the conversion.** Only the CPU backends implement the Fock representation (always on a single thread), and the backend is chosen once for the whole run, so a circuit that reaches the Fock representation anywhere is refused on a GPU backend rather than quietly moved to the CPU.
- **A state spanning several modes cannot be converted.** The modes one state spans are correlated, so they cannot be converted one at a time. A circuit that places such a state and needs the conversion is refused with a `ValueError`; one that stays bosonic runs as usual.
- **A dimension far past the state's own extent is refused.** The conversion loses its accuracy there, and what it would return is not a truncated state, so it is refused rather than answered. Lowering the dimension is the way out, which is the opposite of what a truncation problem would ask for.

(sec:simulation-fock-options)=

### Options for the Fock backend

The attributes below apply only to circuits simulated on the Fock backend, i.e. circuits that run in the Fock representation, whether their placed states are {py:class}`~claveles.circuit.state.FockState` objects or were converted into it as described above. They are ignored for bosonic circuits. Setting any of them and then running a graph or machinery program reports an {py:class}`~claveles.warnings.IgnoredOptionWarning`, since only a circuit reads them.

The Fock backend stores the state as a *matrix product state*, a tensor network whose internal dimension &mdash; the *bond dimension* &mdash; grows as operations build up entanglement between the modes. Capping it keeps the simulation affordable at the cost of discarding the smallest singular values of each bond, which makes the result approximate.

| Attribute | Default | Description |
| --------- | ------- | ----------- |
| {py:attr}`~claveles.client.simulator_client.SimulatorClient.max_bond_dim` | `None` | Upper bound on the bond dimension. `None` leaves it unbounded, so the simulation is exact. |
| {py:attr}`~claveles.client.simulator_client.SimulatorClient.svd_cutoff` | `0.0` | Discard singular values below this threshold. `0.0` disables threshold-based truncation. |
| {py:attr}`~claveles.client.simulator_client.SimulatorClient.track_truncation_error` | `False` | When `True`, record the truncation error of each shot in {py:attr}`~claveles.client.simulator_client.ShotResult.truncation_error`, read alongside that shot's measured values and state (see {ref}`sec:simulation-reading-results`). |
| {py:attr}`~claveles.client.simulator_client.SimulatorClient.max_saved_state_elements` | `2**20` | Upper bound on the number of elements in a single saved state. `None` asks for no bound. Settable for `mode="local"` only &mdash; the Cloud Simulator Server applies a bound of its own. |

With `max_bond_dim=None` and `svd_cutoff=0.0` &mdash; the defaults &mdash; nothing is truncated and the simulation is exact, for mixed states just as for pure ones.

````{note}
**Mixed-state circuits benefit from raising `max_bond_dim` explicitly.**

A mixed state arises once a circuit applies a channel such as {py:class}`std.LossChannel <claveles.circuit.ops.std.LossChannel>`, keeps only some modes with {py:class}`std.PartialTrace <claveles.circuit.ops.std.PartialTrace>`, or starts from a density matrix. The Fock backend then holds the state as a *purification*: it keeps the discarded environment in the state instead of averaging over it, which is what guarantees the density matrix stays physical even under truncation.

Those extra degrees of freedom also make the bond dimension grow faster than in an equivalent unitary circuit. The default is shared with pure circuits and is not raised automatically, so a mixed-state circuit that hits a cap you set for a pure one may lose more accuracy than expected. Raise it, or leave it at `None`, when accuracy matters.
````

Set {py:attr}`~claveles.client.simulator_client.SimulatorClient.track_truncation_error` to see how much was actually discarded. The reported value accumulates the discarded singular-value weight over the whole circuit and is reported per shot, for mixed and pure states alike.

```{note}
**Saved states are not normalized.**

A {py:class}`~claveles.circuit.state.FockState` carries whatever weight the state has left: the norm of {py:meth}`~claveles.circuit.state.FockState.state_vector` for a pure state, the trace of {py:meth}`~claveles.circuit.state.FockState.density_matrix` for a mixed one. The loss stays visible instead of being divided away, and the same circuit reports it identically in either representation. Divide by the norm or the trace when you need a normalized state.

The weight falls below 1 when the initial state was given with a weight below 1, when a finite `cutoff_dim` cuts off a state's Fock tail, or when a `max_bond_dim` cap discards part of it. With none of those in play it stays at 1, to within a few 1e-8 of float32 round-off from transporting the state.

Placed states follow the same convention, which is why `normalize` defaults to `False` on the {py:class}`~claveles.circuit.state.FockState` factories: a weight below 1 is accepted as a truncated state and simulated as given, while a weight above 1 is rejected as an unnormalized input. Pass `normalize=True` to rescale the initial state to unit weight before it is sent; the saved state is still reported unnormalized, so whatever the circuit itself truncates still shows up in it.
```

The last one guards memory rather than accuracy. A saved state is handed back as a dense array, whose size is the product of the modes' cutoff dimensions &mdash; or the square of that for a mixed state, since a density matrix holds `dim` &times; `dim` entries. Both grow exponentially with the number of modes, so a many-mode state cannot be handed back at all and the simulation fails instead of exhausting memory:

```
RuntimeError: The saved Fock state of 11 modes (cutoffs 4x4x4x4x4x4x4x4x4x4x4) exceeds the
1048576-element limit for a single saved state. Apply a partial trace to keep fewer modes, or
raise max_saved_state_elements.
```

Keeping fewer modes with {py:class}`std.PartialTrace <claveles.circuit.ops.std.PartialTrace>` is usually the better answer, since it bounds the size at the source. Raise the limit when you genuinely need the full joint state and have the memory for it. The bound applies per saved state and is only consulted when {py:attr}`~claveles.client.simulator_client.SimulatorClient.state_save_policy` saves something.

```{seealso}
{ref}`sec:gates-channels` for the operations that make a state mixed.
```


## Simulating circuit

You can simulate the circuit with {py:func}`~claveles.execute.execute`.
This function takes a circuit and a client as arguments and returns the execution result.

In [ ]:
from claveles.execute import execute

result = execute(circuit, client)

In [ ]:
type(result.client_result)

{py:class}`~claveles.client.SimulatorClientResult` has the following attributes.

A simulation reports at the level it was given, so exactly one of the three per-representation
attributes below answers and {py:attr}`~claveles.client.SimulatorClientResult.execution_result` is that
one. The others are properties, and reading one raises {py:class}`ValueError` naming what ran: a graph
run has no circuit-level result to report, not an empty one.

{py:attr}`~claveles.client.SimulatorClientResult.shot_results` is refused the same way -- but a graph or
machinery run does report every shot, as measured values on its own result, read with the same
`get_shot_measured_value`, indexing and iteration. Only the state that
{py:class}`~claveles.client.simulator_client.ShotResult` pairs them with is circuit-only.

| Attribute | Type | Description |
| --------- | ---- | ----------- |
| {py:attr}`~claveles.client.SimulatorClientResult.execution_details` | {py:class}`~claveles.client.simulator_client.ExecutionDetails` | Versions and timestamps related to a simulation. |
| {py:attr}`~claveles.client.SimulatorClientResult.circuit_result`   | {py:class}`~claveles.circuit.CircuitResult` | Measurement results after circuit simulation. |
| {py:attr}`~claveles.client.SimulatorClientResult.graph_result`   | {py:class}`~claveles.graph.GraphResult` | Measurement results after graph simulation. |
| {py:attr}`~claveles.client.SimulatorClientResult.machinery_result`   | {py:class}`~claveles.machinery.MachineryResult` | Measurement results after machinery simulation. |
| {py:attr}`~claveles.client.SimulatorClientResult.shot_results`     | list of {py:class}`~claveles.client.simulator_client.ShotResult` | What each shot produced: the values it measured, and the state it left behind. One entry per shot. A shot whose state was not saved has {py:attr}`~claveles.client.simulator_client.ShotResult.state` set to `None`, so {py:attr}`~claveles.client.simulator_client.SimulatorClient.state_save_policy` of `"none"` leaves every state `None` and `"first_only"` leaves all but the first. Available for circuit simulation only. |
| {py:attr}`~claveles.client.SimulatorClientResult.execution_result` | {py:class}`~claveles.circuit.CircuitResult`, {py:class}`~claveles.graph.GraphResult` or {py:class}`~claveles.machinery.MachineryResult` | The result of what was run. Always available, so it is what to read when the code does not depend on which representation that was. |
| {py:attr}`~claveles.client.SimulatorClientResult.wait_time` | `timedelta` or None | The waiting time until the job starts to run. |
| {py:attr}`~claveles.client.SimulatorClientResult.execution_time`  | `timedelta` or None | The time to execute a representation. |
| {py:attr}`~claveles.client.SimulatorClientResult.total_time`  | `timedelta` or None | The total time to execute a representation. |
| {py:attr}`~claveles.client.abstract.AbstractClientResult.n_shots`   | `int` | The number of shots. Inherited from {py:class}`~claveles.client.abstract.AbstractClientResult`, which reads it off {py:attr}`~claveles.client.abstract.AbstractClientResult.execution_result` for every client. |

When you simulate the circuit representation, you can get the measurement result of the circuit representation for any shot by using index access.

In [ ]:
from pprint import pprint

pprint(result[0])  # Get the 0-th shot result

In [ ]:
pprint(result[1])  # Get the 1-st shot result

Each {py:class}`~claveles.circuit.result.CircuitModeMeasuredValue` carries the mode it read as {py:attr}`~claveles.circuit.result.CircuitModeMeasuredValue.mode` and the value as {py:attr}`~claveles.circuit.result.CircuitModeMeasuredValue.value`.

(sec:simulation-reading-results)=

### Reading a shot

A simulation reports each shot as a pair: the values that shot measured, and the state it left behind.
{py:attr}`~claveles.client.SimulatorClientResult.shot_results` holds one entry per shot, whatever the save
policy, so a shot always has a place in the list even when its state was not kept.

In [ ]:
shot = result.client_result.shot_results[0]

print("measured:", shot.measured_value)
print("state   :", shot.state.modes, "->", type(shot.state.state_list[0]).__name__)
print("shots   :", len(result.client_result.shot_results))

Each half answers a different question.

* {py:attr}`~claveles.client.simulator_client.ShotResult.measured_value` is what the measurements read. A
  mode is measured at most once, because measuring consumes it, so
  {py:meth}`~claveles.circuit.result.CircuitShotMeasuredValue.get_value` takes a mode index and the answer is
  unambiguous.
* {py:attr}`~claveles.client.simulator_client.ShotResult.state` is the modes that were *not* measured, as a
  {py:class}`~claveles.circuit.state.QuantumState` covering their original indices. It is `None` when the
  save policy did not ask for this shot, which is why `state_save_policy="none"` leaves every entry
  `None` and `"first_only"` leaves all but the first.
* {py:attr}`~claveles.client.simulator_client.ShotResult.truncation_error` is the accumulated SVD truncation
  error of the shot, or `None` unless
  {py:attr}`~claveles.client.simulator_client.SimulatorClient.track_truncation_error` was enabled.

A circuit that measures every mode leaves a state with no modes at all. That is reported as an empty
{py:class}`~claveles.circuit.state.QuantumState` rather than as `None`, which is how "everything was
measured" stays distinguishable from "no state was saved".

For the state objects themselves, see {ref}`Working with Quantum States <sec:quantum-states-in-the-sdk>`.

In this example, the executed circuit is a teleportation circuit that transfers the state of mode 0 to
mode 2. We can verify whether the {py:attr}`~claveles.circuit.state.QuantumState.mean` of mode 2 in the
final state matches the initial {py:attr}`~claveles.circuit.state.QuantumState.mean` of mode 0.

In [ ]:
error_threshold = 1.0

assert isinstance(result.client_result, SimulatorClientResult)
for shot in result.client_result.shot_results:
    state = shot.state
    assert state is not None
    # Modes 0 and 1 were measured and so are gone: mode 2 is the only one left, which puts its x at
    # index 0 and its p at index 1.
    assert state.modes == [2]
    teleported_x = state.mean[0].real
    teleported_p = state.mean[1].real

    assert abs(teleported_x - mode0_x) < error_threshold
    assert abs(teleported_p - mode0_p) < error_threshold

print("Teleportation successful!")

### Visualizing the output mode

To see the effect of the teleportation protocol, we visualize the Wigner function of the output mode in the first shot.

In [ ]:
client_result = result.client_result
assert isinstance(client_result, SimulatorClientResult)
final_state = client_result.shot_results[0].state
assert final_state is not None
# extract_mode returns a QuantumState covering that mode alone, which is what draws.
final_state.extract_mode(2).draw_wigner(xvec=linspace(-10, 10, 401), pvec=linspace(-10, 10, 401));

Since the initial modes 1 and 2 have finite squeezing levels, output mode 2 differs slightly and randomly from input mode 0.

(sec:simulation-graph-machinery-results)=

## Simulating a graph or machinery program

{py:func}`~claveles.execute.execute` takes any of the three representations, and the result it hands
back is read at the level that was submitted. A circuit reports per **mode**, because a circuit
measures modes; a graph and a machinery program report per **macronode**, because that is what the
machine measures.

| Submitted | Result | One shot | One measured thing |
| --- | --- | --- | --- |
| circuit | {py:class}`~claveles.circuit.CircuitResult` | {py:class}`~claveles.circuit.result.CircuitShotMeasuredValue` | {py:class}`~claveles.circuit.result.CircuitModeMeasuredValue` (one mode) |
| graph | {py:class}`~claveles.graph.GraphResult` | {py:class}`~claveles.graph.result.GraphShotMeasuredValue` | {py:class}`~claveles.graph.result.GraphMacronodeMeasuredValue` (one macronode) |
| machinery | {py:class}`~claveles.machinery.MachineryResult` | {py:class}`~claveles.machinery.result.MachineryShotMeasuredValue` | {py:class}`~claveles.machinery.result.MachineryMacronodeMeasuredValue` (one macronode) |

A macronode is read at four micronodes, and a graph macronode reports the two of them a graph can name:
{py:attr}`~claveles.graph.result.GraphMacronodeMeasuredValue.m_b` and
{py:attr}`~claveles.graph.result.GraphMacronodeMeasuredValue.m_d`. A shot is indexed either by macronode
index or by coordinate:


In [ ]:
from claveles.graph import GraphRepr
from claveles.graph.constant import BLANK_MODE
from claveles.graph.ops import Initialization, Measurement

graph = GraphRepr(2, 2)
graph.place_operation(Initialization((0, 0), 0.0, (BLANK_MODE, 0)))
graph.place_operation(Measurement((0, 1), 0.0))

graph_result = execute(graph, SimulatorClient(n_shots=2)).execution_result
shot = graph_result.get_shot_measured_value(0)

print("measured macronodes (index) :", shot.index_list())
print("measured macronodes (coord) :", shot.coord_list())
print("by coordinate               :", shot[(0, 1)])


Only the macronodes the program marks for readout appear, which is why a two-by-two graph with one
measurement reports one entry rather than four. A machinery result reads the same way, with all four
micronodes ({py:attr}`~claveles.machinery.result.MachineryMacronodeMeasuredValue.m_a` to
{py:attr}`~claveles.machinery.result.MachineryMacronodeMeasuredValue.m_d`) on each entry, since the machinery
representation names all four.

```{seealso}
{ref}`sec:graph-composing-shots` for packing several shots into one graph and splitting the result
back apart.
```


(sec:simulation-resource-and-initialized-states)=

## Resource and initialized states

Except when executing a circuit representation, the simulator only utilizes squeezed states as
resources. The resource modes are used for mode initialization, as explained in
{ref}`the graph.ops.Initialization entry <sec:gate-graph-repr-initialization>`, and the initialized modes are also squeezed states.

Here $s$ is the squeezing level in dB set by
{py:attr}`~claveles.client.simulator_client.SimulatorClient.resource_squeezing_level`.

| Representation | Parameter | Resource | Initialized |
| --- | --- | --- | --- |
| Circuit | — | (no concept of a resource) | the state you placed on the mode |
| Graph | Squeezing level | $s$ dB | $(s-10\log_{10} 2)$ dB |
| Graph | Anti-squeezing level | $s$ dB (the same value) | $(s-10\log_{10} 2)$ dB |
| Graph | Squeezing angle | 0 | $\theta+\pi/2$, where $\theta$ is the parameter of {py:class}`~claveles.graph.ops.Initialization` |
| Machinery | Squeezing level | $s$ dB | (not emphasized; see below) |
| Machinery | Anti-squeezing level | $s$ dB (the same value) | (not emphasized; see below) |
| Machinery | Squeezing angle | 0 | (not emphasized; see below) |

```{seealso}
See {ref}`sec:squeezing-level-and-squeezing-angle` for the definitions of squeezing level and squeezing
angle.
```

The squeezing and anti-squeezing levels agreeing means the simulator's resource mode is a **pure**
state, which is the one place it differs from the same machinery representation executed through
{py:class}`~claveles.client.optqc_client.OptQCClient`, where finite squeezing and optical loss make it mixed
— compare {ref}`sec:execution-resource-and-initialized-states`.

In the **machinery** representation, mode initialization is just one of the operations that can be
realized through homodyne angle configuration, as described in
{ref}`the graph.ops.Initialization entry <sec:gate-graph-repr-initialization>`, so the state after initialization is not treated as something
separate — which is why the "Initialized" column says nothing for it.

In the **graph** representation, the resource mode is the state prior to initialization. Once
initialized, the mode becomes manageable through the operations defined within the graph representation,
and the "Initialized" column gives the parameters of the mode after initialization; that mode is
simplified as a pure state too. Of those parameters, only the squeezing angle is under your control,
through the `theta` parameter of {py:class}`~claveles.graph.ops.Initialization`.

The **circuit** representation has no concept of a resource or of initialization at all: the mode starts
from the {py:class}`~claveles.circuit.state.QuantumState` the circuit places on it, whose components may be
{py:class}`~claveles.circuit.state.BosonicState`, {py:class}`~claveles.circuit.state.GaussianState` or
{py:class}`~claveles.circuit.state.FockState`.